ConvLSTM (Convolutional Long Short-Term Memory) is an advanced deep learning model that combines spatial feature extraction with sequential time-series forecasting

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

I0000 00:00:1790412746.767087  172241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790412746.789251  172241 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790412747.625270  172241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def build_convlstm_model(input_shape, num_layers=2, hidden_dims=[16, 32], kernel_size=(3, 3)):
    """
    Builds a stacked ConvLSTM network.
    input_shape: (sequence_length, height, width, channels)
    num_layers: Number of ConvLSTM layers to stack
    hidden_dims: List containing the number of filters for each layer
    kernel_size: Size of the convolutional window
    """
    model = models.Sequential()
    
    # Input Layer
    model.add(layers.Input(shape=input_shape))
    
    # Hidden ConvLSTM Layers
    for i in range(num_layers):
        # The last ConvLSTM layer usually sets return_sequences=False 
        # if you only want to predict the single next frame. 
        # Set return_sequences=True to get the output history for all time steps.
        return_seq = True if i < num_layers - 1 else False
        
        model.add(layers.ConvLSTM2D(
            filters=hidden_dims[i],
            kernel_size=kernel_size,
            padding='same',
            return_sequences=return_seq,
            activation='tanh',
            recurrent_activation='hard_sigmoid'
        ))
        
        # Batch Normalization helps stabilize training in deep spatiotemporal models
        model.add(layers.BatchNormalization())
        
    return model


In [3]:
# Hyperparameters
batch_size = 2
sequence_length = 5
height, width = 64, 64
channels = 3  # e.g., RGB video frames

In [4]:
# Shape format for TensorFlow: (Sequence_Length, Height, Width, Channels)
input_shape = (sequence_length, height, width, channels)

In [5]:
# Instantiate the model
# Layer 1 outputs 16 filters, Layer 2 outputs 32 filters
model = build_convlstm_model(input_shape, num_layers=2, hidden_dims=[16, 32])


In [6]:
# Compile the model
model.compile(optimizer='adam', loss='mse')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d (ConvLSTM2D)        │ (None, 5, 64, 64, 16)  │        11,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 5, 64, 64, 16)  │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 64, 64, 32)     │        55,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,624 (260.25 KB)

 Trainable params: 66,528 (259.88 KB)

 Non-trainable params: 96 (384.00 B)

In [7]:
# Create dummy data: (Batch, Time, Height, Width, Channels)
import numpy as np
dummy_input = np.random.random((batch_size, sequence_length, height, width, channels)).astype(np.float32)

In [8]:
# Forward pass
output = model.predict(dummy_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step


In [9]:
print(f"\nInput Shape: {dummy_input.shape}")
print(f"Output Shape: {output.shape}")
# Because return_sequences=False on the last layer, 
# the output removes the time dimension: (Batch, Height, Width, Filters)
# Expected output: (2, 64, 64, 32)


Input Shape: (2, 5, 64, 64, 3)
Output Shape: (2, 64, 64, 32)
